# N4: Multilingual Intent Classifier (distilbert-base-multilingual-cased)

- **Base Model**: `distilbert-base-multilingual-cased`
- **Dataset**: 1,500 multilingual queries across 5 classes {eligibility, documents, how_to_apply, status, scam_check}
- **Objective**: CrossEntropyLoss (5-class classification)
- **Target**: Macro F1 >= 0.90 across all 5 languages
- **Seed**: `42` (pinned across Python, NumPy, PyTorch)
- **Hardware**: Kaggle / Colab T4 (16 GB VRAM). Set `SAMPLE_FRAC=0.05` for top-to-bottom smoke test.


In [ ]:
# Pinned versions for reproducible Kaggle T4 execution
!pip install -q transformers==4.47.1 sentence-transformers==3.3.1 peft==0.14.0 trl==0.13.0 bitsandbytes==0.45.0 mlflow==2.19.0 huggingface_hub==0.27.0


In [ ]:
import os, json, random, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
CLASSES = ["eligibility", "documents", "how_to_apply", "status", "scam_check"]
model_id = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=len(CLASSES))
print(f"Initialized {model_id} for 5-class intent classification; seed={SEED}; checkpoint=checkpoints/N4_intent_clf")


In [ ]:
# MLflow logging + Hugging Face Hub push (requires HF_TOKEN env var)
import os, mlflow
mlflow.set_experiment('yojana-mitra-training')
with mlflow.start_run(run_name='N4_intent_clf'):
    mlflow.log_params({'base_model': 'distilbert-base-multilingual-cased', 'seed': 42, 'sample_frac': os.getenv('SAMPLE_FRAC', '0.05')})
if os.getenv('HF_TOKEN'):
    from huggingface_hub import HfApi
    print('Ready to upload checkpoints/N4_intent_clf to yojana-mitra/distilbert-welfare-intent-v1')
